# Sparse linear regression — Boomerang & Sticky Boomerang vs HMC

A moderate-dimensional sparse regression benchmark with known ground-truth
support. The aim is to show that:

1. The plain Boomerang correctly samples the Gaussian-prior posterior
   (verified against the analytical closed-form posterior).
2. The Sticky Boomerang recovers the true support more cleanly than either
   naive HMC (Gaussian prior) or HMC with a horseshoe sparsity prior.
3. PLI thinning matches Brent at this dimension.

This notebook is **deterministic**: running all cells from a clean kernel
should reproduce the exact tables and plots, run after run.


## 0. Config


In [ ]:
# --- Reproducibility ---
SEED = 0

# --- Data: sparse linear regression ---
N            = 2_000      # observations
D            = 200        # features
K_NONZERO    = 10         # true number of signals
SIGNAL_SCALE = 2.0        # signals drawn from N(0, SIGNAL_SCALE^2)
NOISE_STD    = 1.0        # observation noise σ
INTERCEPT_TRUE = 0.0      # no intercept here — keeps the closed-form clean

# --- Prior / likelihood (used by Boomerang, Gaussian NUTS, and analytic ref) ---
PRIOR_STD     = 5.0       # Gaussian-prior slab (wide-ish, lets data speak)
LIK_NOISE_STD = NOISE_STD

# --- Sampler budgets ---
N_SKEL       = 30_000     # PDMP skeleton points
N_RESAMPLE   = 50_000     # path-resampled draws
BURNIN_FRAC  = 0.5
REFRESH_RATE = 1.0
KAPPA_NULL   = 1.0        # stickiness for all coordinates (no intercept here)

# --- NUTS ---
NUTS_DRAWS    = 2_000
NUTS_TUNE     = 1_000
NUTS_CHAINS   = 2
NUTS_TARGET_ACCEPT = 0.95   # higher target_accept is safer for horseshoe

# --- Toggles ---
RUN_HORSESHOE = True       # set False to skip horseshoe HMC (~5-10 min)


In [ ]:
import os, sys, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)

if 'sazz' not in sys.modules:
    try:
        os.chdir('../..')
    except FileNotFoundError:
        pass

import numpy as np
import torch
import matplotlib.pyplot as plt
import pymc as pm
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, auc

from sazz.samplers.AutomaticBoomerangSampler import AutomaticBoomerangSampler
from sazz.samplers.StickyAutomaticBoomerangSampler import StickyAutomaticBoomerangSampler
from sazz.models.make_models import make_linear_regression
from sazz.utils.sampling import resample_boomerang_path, resample_boomerang_path_sticky


def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed(SEED)
print(f"torch={torch.__version__}  numpy={np.__version__}  pymc={pm.__version__}")


## 1. Data

`N` observations of `D` standardised features, `K_NONZERO` true signals
drawn from `N(0, SIGNAL_SCALE²)` placed at random positions; the remaining
`D - K_NONZERO` coefficients are exactly zero. This is the cleanest sparse
DGP — the Sticky Boomerang's spike component matches the truth exactly.


In [ ]:
set_seed(SEED)
rng = np.random.default_rng(SEED)

X = rng.normal(size=(N, D))
beta_true = np.zeros(D)
signal_idx = rng.choice(D, size=K_NONZERO, replace=False)
signal_idx.sort()
beta_true[signal_idx] = SIGNAL_SCALE * rng.normal(size=K_NONZERO)
y = X @ beta_true + INTERCEPT_TRUE + NOISE_STD * rng.normal(size=N)

X = (X - X.mean(0)) / X.std(0)

X_t = torch.tensor(X, dtype=torch.float64)
y_t = torch.tensor(y, dtype=torch.float64)

# Convenience masks
is_signal = beta_true != 0
print(f"N={N}, D={D}, true signals = {K_NONZERO}/{D} "
      f"(SNR ≈ {(np.abs(beta_true[is_signal])).mean() / NOISE_STD:.2f})")
print(f"Signal indices: {signal_idx.tolist()[:10]}{'...' if K_NONZERO > 10 else ''}")


## 2. Analytical posterior (gold standard)

For Gaussian prior `β ~ N(0, σ_p² I)` and Gaussian likelihood
`y | β ~ N(Xβ, σ² I)` the posterior is exactly
`β | y ~ N(μ_post, Σ_post)` with

    Σ_post = (X'X / σ² + I / σ_p²)⁻¹
    μ_post = Σ_post · X' y / σ²

This is the *true* posterior the plain Boomerang and Gaussian-prior NUTS
are sampling. Computing it directly gives us an oracle for correctness
checks that doesn't depend on either approximation.


In [ ]:
prec_post  = X.T @ X / LIK_NOISE_STD ** 2 + np.eye(D) / PRIOR_STD ** 2
Sigma_post = np.linalg.inv(prec_post)
mu_post    = Sigma_post @ (X.T @ y / LIK_NOISE_STD ** 2)
sd_post    = np.sqrt(np.diag(Sigma_post))

print(f"Analytical posterior on β: μ ∈ [{mu_post.min():.2f}, {mu_post.max():.2f}], "
      f"σ ∈ [{sd_post.min():.3f}, {sd_post.max():.3f}]")
print(f"Mean recovery on signals: RMSE = "
      f"{np.sqrt(((mu_post[is_signal] - beta_true[is_signal])**2).mean()):.3f}")


## 3. NUTS, Gaussian prior

The naive Bayesian baseline: same prior the Boomerang uses. With `n > d`
and a wide-ish prior the posterior contracts well on signals; nulls retain
prior-shaped mass around zero.


In [ ]:
set_seed(SEED)

with pm.Model() as model_gauss:
    betas = pm.Normal('betas', mu=0.0, sigma=PRIOR_STD, shape=D)
    pm.Normal('y', mu=X @ betas, sigma=LIK_NOISE_STD, observed=y)

    t0 = time.perf_counter()
    nuts_gauss = pm.sample(
        draws=NUTS_DRAWS, tune=NUTS_TUNE, chains=NUTS_CHAINS,
        target_accept=NUTS_TARGET_ACCEPT, progressbar=False, random_seed=SEED,
    )
    t_nuts_gauss = time.perf_counter() - t0

samples_nuts_gauss = nuts_gauss.posterior['betas'].values.reshape(-1, D)
print(f"NUTS-Gaussian: {samples_nuts_gauss.shape[0]} draws  |  "
      f"wall = {t_nuts_gauss:.2f}s")


## 4. NUTS, horseshoe prior

The proper sparsity baseline. Non-centered parameterisation
(`β = z · λ · τ` with `z ~ N(0,1)`) avoids HMC funnels in the local-shrinkage
parameters. Reasonably slow but the standard choice for sparse Bayesian
regression.

Note: HMC on a horseshoe never produces exact zeros — sparsity is implied
by extreme shrinkage, not by point masses. Support recovery from horseshoe
output requires thresholding (we use the credible-interval rule below).


In [ ]:
if RUN_HORSESHOE:
    set_seed(SEED)

    with pm.Model() as model_hs:
        # Global shrinkage; tau0 ≈ k / (d - k) / sqrt(n)  is a sensible default
        tau0 = K_NONZERO / (D - K_NONZERO) / np.sqrt(N)
        tau    = pm.HalfCauchy('tau',    beta=tau0)
        lambdas = pm.HalfCauchy('lambdas', beta=1.0, shape=D)
        z       = pm.Normal('z', mu=0.0, sigma=1.0, shape=D)
        betas   = pm.Deterministic('betas', z * lambdas * tau)
        pm.Normal('y', mu=X @ betas, sigma=LIK_NOISE_STD, observed=y)

        t0 = time.perf_counter()
        nuts_hs = pm.sample(
            draws=NUTS_DRAWS, tune=NUTS_TUNE, chains=NUTS_CHAINS,
            target_accept=NUTS_TARGET_ACCEPT, progressbar=False, random_seed=SEED,
        )
        t_nuts_hs = time.perf_counter() - t0

    samples_nuts_hs = nuts_hs.posterior['betas'].values.reshape(-1, D)
    print(f"NUTS-horseshoe: {samples_nuts_hs.shape[0]} draws  |  "
          f"wall = {t_nuts_hs:.2f}s")
else:
    samples_nuts_hs = None
    t_nuts_hs = float('nan')
    print("Horseshoe NUTS skipped (set RUN_HORSESHOE=True to enable).")


## 5. Build the four PDMP samplers


In [ ]:
target = make_linear_regression(
    X_t, y_t,
    prior_std=PRIOR_STD,
    intercept_prior_std=PRIOR_STD,    # no intercept; ignored
    noise_std=LIK_NOISE_STD,
)

# Uniform stickiness — no intercept to special-case
kappa = torch.full((target.D,), KAPPA_NULL, dtype=torch.float64)


def build_sampler(sticky: bool, thinning: str):
    cls = StickyAutomaticBoomerangSampler if sticky else AutomaticBoomerangSampler
    kwargs = dict(grad_target=target.grad_target, D=target.D,
                  refresh_rate=REFRESH_RATE, thinning=thinning)
    if sticky:
        kwargs['kappa'] = kappa
    s = cls(**kwargs)
    s.preprocess(x_ref=target.x_ref, Sigma_inv=target.Sigma_inv)
    return s


SAMPLERS = {
    'Boom-Brent'  : ('boom',   'brent'),
    'Boom-PLI'    : ('boom',   'pli'),
    'Sticky-Brent': ('sticky', 'brent'),
    'Sticky-PLI'  : ('sticky', 'pli'),
}


## 6. Run all four PDMP samplers


In [ ]:
def run_one(kind: str, thinning: str):
    set_seed(SEED)
    sampler = build_sampler(sticky=(kind == 'sticky'), thinning=thinning)

    t0 = time.perf_counter()
    res = sampler.sample(N=N_SKEL, diagnostics=False)
    t_sample = time.perf_counter() - t0

    fn = resample_boomerang_path_sticky if kind == 'sticky' else resample_boomerang_path
    samples = fn(
        res['positions'].cpu().numpy(),
        res['velocities'].cpu().numpy(),
        res['times'].cpu().numpy(),
        target.x_ref.cpu().numpy(),
        N_resample=N_RESAMPLE, burnin_frac=BURNIN_FRAC,
    )
    return samples, t_sample, res


results = {}
for name, (kind, thinning) in SAMPLERS.items():
    samples, wall, raw = run_one(kind, thinning)
    results[name] = dict(samples=samples, wall=wall, raw=raw, kind=kind,
                         thinning=thinning)
    print(f"{name:<14}  wall={wall:6.2f}s  resampled={samples.shape[0]}")


## 7. Wall-clock summary


In [ ]:
walls = {'NUTS-Gaussian': t_nuts_gauss}
if RUN_HORSESHOE:
    walls['NUTS-horseshoe'] = t_nuts_hs
walls.update({n: r['wall'] for n, r in results.items()})

print(f"{'sampler':<16} {'wall (s)':>9}")
print('-' * 26)
for name, w in walls.items():
    print(f"{name:<16} {w:>9.2f}")


## 8. Correctness check vs analytical posterior

A pure sanity check for the *plain* Boomerang variants. Posterior means
and stds should match the analytical Gaussian posterior to within MC error.
This is independent of any HMC comparison.


In [ ]:
print(f"{'sampler':<14}  {'max |Δμ|':>10}  {'max |Δσ|':>10}  "
      f"{'mean |Δμ|':>10}  {'mean |Δσ|':>10}")
print('-' * 65)
for name in ['Boom-Brent', 'Boom-PLI']:
    s = results[name]['samples'][:, 1:]   # drop intercept column
    mu = s.mean(0); sd = s.std(0)
    d_mu = np.abs(mu - mu_post)
    d_sd = np.abs(sd - sd_post)
    print(f"{name:<14}  {d_mu.max():>10.4f}  {d_sd.max():>10.4f}  "
          f"{d_mu.mean():>10.4f}  {d_sd.mean():>10.4f}")
print()
print("(Healthy values: max |Δμ| < 0.05, mean |Δμ| < 0.01 at this N_RESAMPLE.)")

## 9. Coefficient plots

Two panels:

- **Signals (top):** errorbars (μ ± 2σ) for each true-nonzero coordinate.
  All methods compared side-by-side.
- **Nulls (bottom):** posterior means scattered across the `D - K`
  coordinates that are truly zero. A method that correctly identifies
  these as null should produce a tight cloud near `y = 0`; horseshoe will
  shrink hard but not exactly zero, while Sticky will literally produce
  `μ = 0` for coordinates whose `P(=0) > 0.5` (mode-at-zero).


In [ ]:
def collect_methods():
    methods = [
        ('Analytic',      mu_post,                    sd_post,                    'k',  'D'),
        ('NUTS-Gaussian', samples_nuts_gauss.mean(0), samples_nuts_gauss.std(0),  'C2', 'D'),
    ]
    if RUN_HORSESHOE:
        methods.append(('NUTS-horseshoe', samples_nuts_hs.mean(0),
                        samples_nuts_hs.std(0), 'C4', 'P'))
    boom_styles = {
        'Boom-Brent':   ('C0', 'o'),
        'Boom-PLI':     ('C9', 'v'),
        'Sticky-Brent': ('C1', 's'),
        'Sticky-PLI':   ('C3', '^'),
    }
    for name, (col, marker) in boom_styles.items():
        s = results[name]['samples'][:, 1:]   # drop intercept
        methods.append((name, s.mean(0), s.std(0), col, marker))
    return methods

methods = collect_methods()

# --- Top panel: signals only ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8),
                                gridspec_kw={'height_ratios': [1.2, 1]})
n_methods = len(methods)
offsets = np.linspace(-0.35, 0.35, n_methods)
sig_idx = np.where(is_signal)[0]
xs = np.arange(len(sig_idx))

for (name, mu, sd, color, marker), off in zip(methods, offsets):
    ax1.errorbar(xs + off, mu[sig_idx], yerr=2 * sd[sig_idx], fmt=marker,
                 color=color, label=name, capsize=2, markersize=4, lw=1)
ax1.scatter(xs, beta_true[sig_idx], marker='x', color='red', s=60,
            linewidths=2, label='true', zorder=5)
ax1.axhline(0, color='grey', lw=0.5)
ax1.set_xticks(xs)
ax1.set_xticklabels([f'β_{i}' for i in sig_idx], rotation=45, ha='right')
ax1.set_ylabel('coefficient value')
ax1.set_title(f'Signal coefficients ({K_NONZERO} of {D}) — posterior μ ± 2σ')
ax1.legend(loc='best', frameon=False, fontsize=8, ncol=3)

# --- Bottom panel: nulls as scatter ---
null_idx = np.where(~is_signal)[0]
for (name, mu, sd, color, marker) in methods:
    ax2.scatter(null_idx, mu[null_idx], color=color, marker=marker, s=15,
                alpha=0.6, label=name)
ax2.axhline(0, color='red', lw=1, label='truth (=0)')
ax2.set_xlabel('coefficient index')
ax2.set_ylabel('posterior mean')
ax2.set_title(f'Null coefficients ({D - K_NONZERO} of {D}) — posterior means')
ax2.legend(loc='best', frameon=False, fontsize=8, ncol=3)

plt.tight_layout()
plt.show()


## 10. Variable selection

Two related but distinct diagnostics:

- **Inclusion-probability ROC (Sticky only):** treat `1 - P(=0)` as a
  score for "this coordinate is a true signal". Sweep the threshold and
  trace the ROC curve. AUC = 1 means perfect separation; AUC = 0.5 means
  the inclusion probability is uninformative.
- **Support recovery F1 at 95% credible-interval rule (all methods):**
  declare a coordinate a signal if its 95% posterior credible interval
  excludes zero. This is the only fair comparison rule across Sticky
  (which has exact zeros), horseshoe (which doesn't), and Gaussian-prior
  HMC (which also doesn't).


In [ ]:
# --- ROC curves for the Sticky variants ---
fig, ax = plt.subplots(figsize=(6, 5))
for name in ['Sticky-Brent', 'Sticky-PLI']:
    s = results[name]['samples'][:, 1:]                # drop intercept
    p_zero = (np.abs(s) < 1e-8).mean(0)
    incl_prob = 1 - p_zero
    fpr, tpr, _ = roc_curve(is_signal.astype(int), incl_prob)
    ax.plot(fpr, tpr, label=f'{name}  (AUC = {auc(fpr, tpr):.3f})', lw=1.5)

ax.plot([0, 1], [0, 1], color='grey', lw=0.5, ls='--', label='chance')
ax.set_xlabel('false positive rate')
ax.set_ylabel('true positive rate')
ax.set_title('Inclusion-probability ROC for Sticky variants')
ax.legend(loc='lower right', frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
def support_via_credible_interval(samples: np.ndarray, alpha: float = 0.05):
    lo = np.quantile(samples, alpha / 2,     axis=0)
    hi = np.quantile(samples, 1 - alpha / 2, axis=0)
    return (lo > 0) | (hi < 0)


def support_via_pzero(samples: np.ndarray, threshold: float = 0.5):
    p_zero = (np.abs(samples) < 1e-8).mean(0)
    return p_zero <= threshold


print(f"{'method':<18}  {'rule':<22}  {'TP':>3}  {'FP':>3}  {'FN':>3}  "
      f"{'P':>5}  {'R':>5}  {'F1':>5}")
print('-' * 78)


def report(name: str, sel: np.ndarray, rule: str):
    tp = int(((sel) & is_signal).sum())
    fp = int(((sel) & ~is_signal).sum())
    fn = int((~sel & is_signal).sum())
    p  = precision_score(is_signal, sel, zero_division=0)
    r  = recall_score(is_signal, sel, zero_division=0)
    f1 = f1_score(is_signal, sel, zero_division=0)
    print(f"{name:<18}  {rule:<22}  {tp:>3}  {fp:>3}  {fn:>3}  "
          f"{p:>5.2f}  {r:>5.2f}  {f1:>5.2f}")


report('NUTS-Gaussian', support_via_credible_interval(samples_nuts_gauss), '95% CI excludes 0')
if RUN_HORSESHOE:
    report('NUTS-horseshoe', support_via_credible_interval(samples_nuts_hs), '95% CI excludes 0')
for name in SAMPLERS:
    s = results[name]['samples'][:, 1:]                # drop intercept
    report(name, support_via_credible_interval(s), '95% CI excludes 0')

print()
for name in ['Sticky-Brent', 'Sticky-PLI']:
    s = results[name]['samples'][:, 1:]
    report(name, support_via_pzero(s), 'P(=0) ≤ 0.5')

## 11. Predictive performance on a held-out test set

Same DGP as training, fresh `(X_test, y_test)` of the same size, and we
evaluate **predictive RMSE marginalised over the full posterior**.
Lower is better.


In [ ]:
test_rng = np.random.default_rng(SEED + 1)
X_test_raw = test_rng.normal(size=(N, D))
X_test = (X_test_raw - X_test_raw.mean(0)) / X_test_raw.std(0)
y_test_clean = X_test @ beta_true + INTERCEPT_TRUE


def predictive_rmse(samples: np.ndarray) -> float:
    preds = samples @ X_test.T              # (n_draws, N_test)
    pred_mean = preds.mean(axis=0)          # marginalised
    return float(np.sqrt(((pred_mean - y_test_clean) ** 2).mean()))


def beta_rmse(samples_or_mean) -> float:
    """RMSE between posterior-mean coefficients and true coefficients.
    Accepts either a sample array or a precomputed mean vector."""
    mu = samples_or_mean.mean(0) if samples_or_mean.ndim == 2 else samples_or_mean
    return float(np.sqrt(((mu - beta_true) ** 2).mean()))

all_methods = [
    ('Analytic',      mu_post,             False),
    ('NUTS-Gaussian', samples_nuts_gauss,  False),
]
if RUN_HORSESHOE:
    all_methods.append(('NUTS-horseshoe', samples_nuts_hs, False))
all_methods += [(n, r['samples'][:, 1:], r['kind'] == 'sticky')
                for n, r in results.items()]                         # drop intercept

print(f"{'method':<16}  {'RMSE on β':>10}  {'Pred. RMSE':>11}  "
      f"{'F1 (95% CI)':>12}  {'P(=0) on nulls':>16}")
print('-' * 78)
for name, obj, sticky in all_methods:
    if obj.ndim == 1:                       # analytic mean only
        b_rmse = beta_rmse(obj)
        p_rmse = float(np.sqrt(((X_test @ obj - y_test_clean) ** 2).mean()))
        analytic_draws = np.random.default_rng(SEED).multivariate_normal(
            mu_post, Sigma_post, size=2000)
        f1 = f1_score(is_signal,
                      support_via_credible_interval(analytic_draws),
                      zero_division=0)
        p0nl_str = f"{'—':>16}"
    else:
        b_rmse = beta_rmse(obj)
        p_rmse = predictive_rmse(obj)
        sel    = support_via_credible_interval(obj)
        f1     = f1_score(is_signal, sel, zero_division=0)
        if sticky:
            p_zero = (np.abs(obj) < 1e-8).mean(0)
            p0nl_str = f"{p_zero[~is_signal].mean():>16.2f}"
        else:
            p0nl_str = f"{'—':>16}"
    print(f"{name:<16}  {b_rmse:>10.4f}  {p_rmse:>11.4f}  "
          f"{f1:>12.2f}  {p0nl_str}")

## 12. Reading the table

What "good" looks like:

- **Analytic** is the literal posterior under a Gaussian prior — both
  Boom variants should match it on β-RMSE and predictive RMSE.
- **NUTS-Gaussian** should also match Analytic — it's a sanity check that
  NUTS converged.
- **NUTS-horseshoe** should beat NUTS-Gaussian on β-RMSE (continuous
  shrinkage is real) but won't get exact zeros.
- **Sticky-Brent / Sticky-PLI** should be competitive or better on β-RMSE,
  and meaningfully better on F1 — the spike-and-slab posterior has actual
  point masses where the truth has them. Predictive RMSE is the closest
  to a "fair fight" with horseshoe; ideally Sticky wins by a small margin
  thanks to cleaner null zeroing.

If Sticky's β-RMSE is worse than horseshoe on signals, that's the slab
shrinking too aggressively — try a wider `PRIOR_STD` or a smaller
`KAPPA_NULL`.
